In [2]:
#!/usr/bin/env python3
import csv, random, datetime
from typing import List, Dict, Any

# ----- Config -----
REGIONS = ['us-east', 'us-west', 'europe', 'asia', 'oceania', 'south-america', 'middle-east', 'africa']
REGION_WEIGHTS = [0.25, 0.20, 0.15, 0.15, 0.10, 0.05, 0.05, 0.05]  

WORLD_CATEGORIES = ['Social', 'Gaming', 'Education', 'Art & Creative', 'Music & Dance',
                    'Sports & Fitness', 'Adventure', 'Relaxation', 'Horror', 'Puzzle',
                    'Roleplay', 'Comedy', 'Simulation', 'Strategy', 'Sci-Fi', 'Fantasy']
WORLD_CATEGORY_WEIGHTS = [0.15, 0.2, 0.1, 0.05, 0.1, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05]
TAGS = ['multiplayer', 'singleplayer', 'competitive', 'casual', 'story-driven',
        'sandbox', 'educational', 'meditation', 'fitness', 'music', 'art',
        'scary', 'funny', 'realistic', 'fantasy', 'sci-fi', 'retro',
        'minimalist', 'social', 'creative', 'immersive', ]
TAG_WEIGHTS = [0.15, 0.1, 0.1, 0.05, 0.1, 0.1, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05]

LANGUAGES = ['en', 'es', 'fr', 'de', 'ja', 'ko', 'zh', 'pt', 'ru', 'it', 'ar', 'hi']
LANGUAGE_WEIGHTS = [0.4, 0.2, 0.08, 0.07, 0.05, 0.05, 0.05, 0.05, 0.03, 0.02, 0.005, 0.005]

AGE_GROUPS = ['13-17', '18-24', '25-34', '35-44', '45-54', '55+']
AGE_WEIGHTS = [0.05, 0.35, 0.35, 0.15, 0.07, 0.03]
ACHIEVEMENT_TYPES = ['Explorer', 'Socializer', 'Creator', 'Competitor',
                     'Collector', 'Helper', 'Pioneer', 'Master', 'Veteran', 'Architect']
ACHIEVEMENT_TYPE_WEIGHTS = [0.2, 0.2, 0.2, 0.1, 0.1, 0.05, 0.05, 0.05, 0.025, 0.025]


AGE_WORLD_WEIGHTS = {
    '13-17': {
        'Social': 0.10, 'Gaming': 0.15, 'Education': 0.05, 'Art & Creative': 0.06,
        'Music & Dance': 0.07, 'Sports & Fitness': 0.06, 'Adventure': 0.12,
        'Relaxation': 0.04, 'Horror': 0.07, 'Puzzle': 0.05, 'Roleplay': 0.07,
        'Comedy': 0.05, 'Simulation': 0.03, 'Strategy': 0.03, 'Sci-Fi': 0.04, 'Fantasy': 0.06
    },
    '18-24': {
        'Social': 0.12, 'Gaming': 0.14, 'Education': 0.06, 'Art & Creative': 0.07,
        'Music & Dance': 0.08, 'Sports & Fitness': 0.07, 'Adventure': 0.10,
        'Relaxation': 0.04, 'Horror': 0.06, 'Puzzle': 0.05, 'Roleplay': 0.06,
        'Comedy': 0.05, 'Simulation': 0.04, 'Strategy': 0.04, 'Sci-Fi': 0.06, 'Fantasy': 0.06
    },
    '25-34': {
        'Social': 0.15, 'Gaming': 0.12, 'Education': 0.07, 'Art & Creative': 0.07,
        'Music & Dance': 0.06, 'Sports & Fitness': 0.06, 'Adventure': 0.07,
        'Relaxation': 0.06, 'Horror': 0.04, 'Puzzle': 0.05, 'Roleplay': 0.05,
        'Comedy': 0.05, 'Simulation': 0.05, 'Strategy': 0.05, 'Sci-Fi': 0.05, 'Fantasy': 0.05
    },
    '35-44': {
        'Social': 0.18, 'Gaming': 0.08, 'Education': 0.08, 'Art & Creative': 0.07,
        'Music & Dance': 0.05, 'Sports & Fitness': 0.05, 'Adventure': 0.05,
        'Relaxation': 0.10, 'Horror': 0.02, 'Puzzle': 0.06, 'Roleplay': 0.04,
        'Comedy': 0.05, 'Simulation': 0.05, 'Strategy': 0.06, 'Sci-Fi': 0.03, 'Fantasy': 0.03
    },
    '45-54': {
        'Social': 0.20, 'Gaming': 0.05, 'Education': 0.10, 'Art & Creative': 0.08,
        'Music & Dance': 0.05, 'Sports & Fitness': 0.05, 'Adventure': 0.04,
        'Relaxation': 0.12, 'Horror': 0.01, 'Puzzle': 0.07, 'Roleplay': 0.03,
        'Comedy': 0.05, 'Simulation': 0.06, 'Strategy': 0.06, 'Sci-Fi': 0.02, 'Fantasy': 0.01
    },
    '55+': {
        'Social': 0.25, 'Gaming': 0.03, 'Education': 0.12, 'Art & Creative': 0.08,
        'Music & Dance': 0.05, 'Sports & Fitness': 0.03, 'Adventure': 0.02,
        'Relaxation': 0.15, 'Horror': 0.00, 'Puzzle': 0.08, 'Roleplay': 0.01,
        'Comedy': 0.05, 'Simulation': 0.07, 'Strategy': 0.05, 'Sci-Fi': 0.01, 'Fantasy': 0.00
    }
}


class VRDataGenerator:
    def __init__(self):
        self.users: List[Dict[str, Any]] = []
        self.worlds: List[Dict[str, Any]] = []

    # ---------- Helpers ----------
    def generate_user_id(self) -> str:
        return f"U{random.randint(100000, 999999)}"
    def generate_connection_id(self) -> str:
        return f"C{random.randint(100000, 999999)}"

    def generate_world_id(self) -> str:
        return f"W{random.randint(1000000, 9999999)}"

    def generate_datetime(self, days_back: int = 365) -> str:
        now = datetime.datetime.now()
        delta = datetime.timedelta(days=random.randint(0, days_back),
                                   hours=random.randint(0, 23),
                                   minutes=random.randint(0, 59))
        return (now - delta).strftime('%Y-%m-%d %H:%M:%S')

    def generate_username(self) -> str:
        adjectives = ['Cool', 'Epic', 'Mystic', 'Shadow', 'Neon', 'Cyber', 'Dark', 'Wild', 'Hyper']
        nouns = ['Player', 'Gamer', 'Explorer', 'Creator', 'Wizard', 'Dragon', 'Ninja', 'Hero', 'Master']
        return f"{random.choice(adjectives)}{random.choice(nouns)}{random.randint(1, 9999)}"

    def generate_world_name(self) -> str:
        prefixes = ['The', 'Virtual', 'Meta', 'Neon', 'Quantum', 'Crystal', 'Dream', 'Galactic', 'Mystic']
        types = ['Arena', 'World', 'Realm', 'Hub', 'Oasis', 'Galaxy', 'Dungeon', 'Island', 'Kingdom']
        return f"{random.choice(prefixes)} {random.choice(types)}{random.randint(1, 9999)}"

    # ---------- Users ----------
    def generate_users(self, count: int = 300) -> List[Dict[str, Any]]:
        users = []
        for _ in range(count):
            users.append({
                'user_id': self.generate_user_id(),
                'user_name': self.generate_username(),
                'is_creator': random.choices(['true', 'false'],weights=[0.01,0.99], k=1)[0],
                'region': random.choices(REGIONS,weights =REGION_WEIGHTS, k=1)[0],
                'preferred_language': random.choices(LANGUAGES, weights =LANGUAGE_WEIGHTS, k=1 )[0],
                'age_group': random.choices(AGE_GROUPS, weights =AGE_WEIGHTS, k=1 )[0],
                'content_filter_level': random.choices(['none', 'low', 'medium', 'high'],weights =[0.4,0.2,0.2,0.2],k=1)[0],
                'preferred_categories': ','.join(random.sample(WORLD_CATEGORIES, k=random.randint(1, 3)))
            })
        self.users = users
        return users

    # ---------- Worlds ----------
    def generate_worlds(self, count: int = 80) -> List[Dict[str, Any]]:
        if not self.users:
            raise ValueError("Users must be generated first.")
        creator_pool = [u for u in self.users if u['is_creator'] == 'true'] or self.users

        worlds = []
        for _ in range(count):
            creator = random.choice(creator_pool)
            worlds.append({
                'world_id': self.generate_world_id(),
                'world_name': self.generate_world_name(),
                'creator_name': creator['user_name'],
                'category': random.choices(WORLD_CATEGORIES,weights= WORLD_CATEGORY_WEIGHTS,k=1)[0],
                'sub_category': random.choice(['Featured', 'Popular', 'New', 'Classic']),
                'tags': random.choices(TAGS,weights= TAG_WEIGHTS,k=3),
                'avg_session_minutes': round(random.uniform(1, 120), 1),
                # 'total_visits': random.randint(0, 1000000),
                'rating': round(random.uniform(1, 5), 1),
                'total_ratings': random.randint(0, 10000),
                'likes': random.randint(0, 50000),
                'max_players': random.choice([4, 8, 16, 32, 50, 100]),
                'current_players': random.randint(0, 50),
                'maturity_rating': random.choices(['E', 'E10+', 'T', 'M'],weights = [0.3,0.3,0.2,0.2], k=1)[0],
                'is_featured': random.choice(['true', 'false', 'false'])
            })
        self.worlds = worlds
        return worlds



    

    def generate_time_spent(self, count: int = 20000) -> List[Dict[str, Any]]:
        if not self.users or not self.worlds:
            raise ValueError("Users & Worlds must be generated first.")
    
        rows, seen = [], set()
    
        # Hệ số để scale từ weight sang phút
        # (có thể tinh chỉnh để dữ liệu trông hợp lý hơn)
        BASE_MIN = 10      # thời gian tối thiểu
        BASE_MAX = 240     # thời gian tối đa
    
        while len(rows) < count:
            user = random.choice(self.users)
            world = random.choice(self.worlds)
    
            user_id = user['user_id']
            world_id = world['world_id']
            age_group = user.get('age_group')
            category = world.get('category')
    
            key = (user_id, world_id)
            if key in seen:
                continue
    
            # mặc định random nếu không tìm thấy weight
            weight = 0.05
            if age_group in AGE_WORLD_WEIGHTS:
                weight = AGE_WORLD_WEIGHTS[age_group].get(category, 0.05)
    
            # map weight (0-0.25) sang khoảng phút
            # scale tuyến tính rồi cộng thêm biến thiên ngẫu nhiên ±10%
            scaled = BASE_MIN + (BASE_MAX - BASE_MIN) * (weight / 0.25)  # 0.25 ~ weight max
            noise = random.uniform(0.9, 1.1)
            avg_time = round(scaled * noise, 1)
    
            rows.append({
                'user_id': user_id,
                'world_id': world_id,
                'avg_time_minutes': avg_time
            })
            seen.add(key)
    
        return rows


    

        

    # ---------- Actions ----------
    def generate_user_actions(self, count: int = 3000) -> List[Dict[str, Any]]:
        if not self.users or not self.worlds:
            raise ValueError("Users & Worlds must be generated first.")
        actions = []
        action_types = ['world_join', 'world_leave', 'world_like', 'world_rate',
                        'world_create', 'world_report', 'item_purchase',
                        'achievement_unlock']
    
        for _ in range(count):
            atype = random.choices(action_types, weights=[0.4,0.4,0.1,0.05,0.01,0.01,0.01,0.02], k=1)[0]
    
            # gán action_value numeric theo ý bạn
            if atype == 'world_rate':
                aval = str(random.randint(1,5))
            elif atype == 'world_like':
                aval = 'true'
            elif atype == 'world_join':
                aval = 'joined'
            elif atype == 'world_leave':
                aval = 'left'
            # elif atype == 'world_create':
            #     aval = 'create'
            elif atype == 'world_report':
                aval = random.choices(['spam','abuse','other'],weights = [0.6,0.3,0.1],k=1)[0]
            elif atype == 'item_purchase':
                aval = random.choices(['cheap','normal','expensive'],weights = [0.6,0.3,0.1],k=1)[0]
            elif atype == 'achievement_unlock':
                aval = random.choices(['common', 'uncommon', 'rare', 'epic', 'legendary'], weights = [0.6,0.2,0.1,0.07,0.03],k=1)[0]
            else:
                aval = ''

    
            actions.append({
                'action_id': f"A{random.randint(100000,999999)}",
                'user_id': random.choice(self.users)['user_id'],
                'world_id': random.choice(self.worlds)['world_id'],
                'action_type': atype,
                'action_timestamp': self.generate_datetime(90),
                'action_value': aval
            })
        return actions


    # # ---------- Achievements ----------
    # def generate_achievements(self, count: int = 1000) -> List[Dict[str, Any]]:
    #     if not self.users:
    #         raise ValueError("Users must be generated first.")
    #     achievements = []
    #     names = ['First Steps', 'World Explorer', 'Social Butterfly', 'Champion',
    #                  'Collector', 'Master Creator', 'Veteran', 'Architect',
    #                  'Speed Runner', 'Puzzle Master', 'Survivor', 'Story Teller']
    #     for _ in range(count):
    #         achievements.append({
    #             'achievement_id': f"H{random.randint(100000, 999999)}",
    #             'user_id': random.choice(self.users)['user_id'],
    #             'achievement_name': random.choice(names),
    #             'achievement_type': random.choices(ACHIEVEMENT_TYPES, weights = ACHIEVEMENT_TYPE_WEIGHTS, k =1),
    #             'world_id': random.choice(self.worlds)['world_id'] if self.worlds else '',
    #             'earned_date': self.generate_datetime(365),
    #             'rarity': random.choices(['common', 'uncommon', 'rare', 'epic', 'legendary'], weights = [0.6,0.2,0.1,0.07,0.03],k=1),
    #             'xp_reward': random.randint(10, 1000)
    #         })
    #     return achievements

    def generate_friend_connections(self, count: int = 1000) -> List[Dict[str, Any]]:
        if not self.users or len(self.users) < 2:
            raise ValueError("Users must be generated first and have at least 2 users.")
        
        connections = []
        for _ in range(count):
            # bốc user1
            user1 = random.choice(self.users)['user_id']
            # bốc user2 khác user1
            user2 = random.choice(self.users)['user_id']
            while user2 == user1:
                user2 = random.choice(self.users)['user_id']
    
            connections.append({
                'connection_id': self.generate_connection_id(),
                'user_id_1': user1,
                'user_id_2': user2,
                'connection_date': self.generate_datetime(365),
                'connection_strength': random.randint(1, 100),
                'mutual_friends_count': random.randint(0, 50),
                'interactions_count': random.randint(0, 1000),
                'last_interaction': self.generate_datetime(7),
                'favorite_world_together': random.choice(self.worlds)['world_id'] if self.worlds else '',
                'total_time_together_hours': round(random.uniform(0, 500), 1)
            })
    
        return connections

    # ---------- CSV ----------
    def save_to_csv(self, filename: str, data: List[Dict[str, Any]]):
        if not data:
            print(f"Empty data for {filename}")
            return
        with open(filename, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=data[0].keys())
            writer.writeheader()
            writer.writerows(data)
        print(f"✓ Saved {len(data)} rows to {filename}")

def main():
    gen = VRDataGenerator()
    users = gen.generate_users(5000)
    gen.save_to_csv('users.csv', users)

    worlds = gen.generate_worlds(400)
    gen.save_to_csv('vrworld.csv', worlds)

    actions = gen.generate_user_actions(40000)
    gen.save_to_csv('user_actions.csv', actions)

    time_spent = gen.generate_time_spent(20000)
    gen.save_to_csv('time_spent.csv', time_spent)


    # achievements = gen.generate_achievements(3000)
    # gen.save_to_csv('achievements.csv', achievements)

    connections = gen.generate_friend_connections(10000)
    gen.save_to_csv('connections.csv', connections)
if __name__ == "__main__":
    main()


✓ Saved 5000 rows to users.csv
✓ Saved 400 rows to vrworld.csv
✓ Saved 40000 rows to user_actions.csv
✓ Saved 20000 rows to time_spent.csv
✓ Saved 10000 rows to connections.csv
